# pytorch で深層学習　演習とminibatch学習

<div style="text-align: right;">
2022/08/01 中山将伸作成<BR>
</div>
    
1) Li-Fデータを使って評価<BR>
2) minibatch学習<BR>

In [ ]:
! pip install torch
! wget https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part2/20161017_LiF_RDF_Voronoi_ME_CE_wMatminer.csv

In [ ]:
import numpy as np
import torch
from torch import nn, optim
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm  



# 1. Li-Fデータ(sample数1500件超)を用いて演習

In [ ]:
path=str("./20161017_LiF_RDF_Voronoi_ME_CE_wMatminer.csv")
df_original=pd.read_csv(path, index_col=0)

df = df_original.dropna()

drop_columns = ['ID','pretty_formula','Cohesive_Energy','Decomposition_Energy','Decomp_UorS','MEvalues_type1','ME_F/S','ME_F/M/S']
df_descriptor = df.drop(drop_columns, axis=1)
target_columns = ['MEvalues_type1']
df_target = df[target_columns]

display(df_descriptor)
display(df_target)


■　保留法　test_size=0.3 として、 X, t -->  train_X, test_X, train_t, test_t  を作成せよ

In [ ]:
X=df_descriptor.values
t=df_target.values

train_X, test_X, train_t, test_t = train_test_split(X,t,test_size=0.3)
print (len(train_X), len(test_X))

■　numpy -->  Tensor形式変換  (train_X --> train_X のまま)

In [ ]:
train_X= torch.Tensor(train_X)
train_t= torch.Tensor(train_t)
test_X= torch.Tensor(test_X)
test_t= torch.Tensor(test_t)

train_X.shape

■　modelの定義  1784の記述子を徐々に 1つのノードに集約させること。

In [ ]:
model= nn.Sequential(
    nn.Linear(1784,128), 
    nn.Sigmoid(),
    nn.Linear(128,16),
    nn.Sigmoid(),
    nn.Linear(16,1),
)

■　Loss関数の定義。 MSE (mean square error)

In [ ]:
loss_fn = nn.MSELoss()   #Loss関数の定義　この場合は MSE


In [ ]:
#optimizer = optim.SGD(model.parameters(), lr=0.001)  #model.parameters() 学習させたい変数　　ln は学習率
optimizer = optim.Adam(model.parameters(), lr=0.001)  #model.parameters() 学習させたい変数　　ln は学習率

■　DNNの実行

In [ ]:
model.train()   # modelをtraining モードにする
loss_history=[]      #  lossのステップごとの推移
losstest_history=[]
for epoch in tqdm(range(200)):
    optimizer.zero_grad()   #optimizerの初期化
    train_y=model(train_X)   #train_y と train_tは異なることに注意
    test_y=model(test_X)
    loss=loss_fn(train_y,train_t)   #Loss関数の計算
    losstest=loss_fn(test_y,test_t)
    loss_history.append(float(loss))
    losstest_history.append(float(losstest))
    loss.backward()   #傾きを計算
    optimizer.step()   #更新処理実施

plt.plot(loss_history)  #blue
plt.plot(losstest_history) #orange

In [ ]:
print (loss_fn(train_y,train_t))
plt.plot(train_y.detach().numpy(),train_t.detach().numpy(),'x')
test_y=model(test_X)
plt.plot(test_y.detach().numpy(),test_t.detach().numpy(),'+')

# 2. minibatch, BatchNorm1d

![fig003.png](https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part2/figs/fig3.png)
<BR>
    
    
![fig004.png](https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part2/figs/fig4.png)


Modelの再定義

BatchNorm1d() 数値の規格化処理を実施 　回帰分析の結果をかなり良くすることがある。  （いつも有効とは限らないことに注意）<BR>
ReLU()  活性化関数処理<BR>

In [ ]:
model= nn.Sequential(
    nn.Linear(1784,128),
    #nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Linear(128,64),
    #nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Linear(64,16),
    #nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Linear(16,1),
)

Minibatchの利用 loader

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(train_X, train_t)
loader = DataLoader(dataset,batch_size=64, shuffle=True)

In [ ]:
#optimizer = optim.SGD(model.parameters(), lr=0.001)  #model.parameters() 学習させたい変数　　ln は学習率
optimizer = optim.Adam(model.parameters(), lr=0.001)  #model.parameters() 学習させたい変数　　ln は学習率

In [ ]:
loss_history=[]
losstest_history=[]

for epoch in tqdm(range(200)):
    model.train()
    for X, t in loader:
        optimizer.zero_grad()   #optimizerの初期化
        y=model(X)
        loss=loss_fn(y,t)
        loss.backward()   #傾きを計算
        optimizer.step()   #更新処理実施
        
    loss_history.append(float(loss))
    test_y=model(test_X)
    losstest=loss_fn(test_y,test_t)
    losstest_history.append(float(losstest))



plt.plot(loss_history)  #blue
plt.plot(losstest_history) #orange


In [ ]:
train_y=model(train_X)

print (loss_fn(train_y,train_t))
plt.plot(train_y.detach().numpy(),train_t.detach().numpy(),'+')

test_y=model(test_X)
print (loss_fn(test_y,test_t))
plt.plot(test_y.detach().numpy(),test_t.detach().numpy(),'x')

In [ ]:
torch.save(model, 'prediction.pth')

In [ ]:
model2=torch.load('prediction.pth', weights_only=False)

In [ ]:
model2